In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# トランスパイルのデフォルト設定と構成オプション


<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは、以下の要件を用いて開発されました。
これらのバージョン以上を使用することを推奨します。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
抽象回路は、QPU がサポートするベースゲートの種類が限られており、任意の操作を実行できないため、トランスパイルが必要です。Transpiler の役割は、任意の回路を指定した QPU 上で実行できるように変換することです。これは、回路をサポートされているベースゲートに変換し、回路の接続性が QPU のものと一致するように必要に応じて SWAP ゲートを導入することで行われます。

[パスマネージャーを使ったトランスパイル](transpile-with-pass-managers) で説明されているように、[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 関数を使って [パスマネージャー](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) を作成し、回路または回路のリストをその [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) メソッドに渡してトランスパイルできます。`generate_preset_pass_manager` を呼び出す際は、最適化レベルと Backend のみを指定して他のオプションにはデフォルト値を使用することも、追加の引数を渡してトランスパイルを細かく調整することもできます。

## パラメーターを指定しない基本的な使い方
この例では、追加のパラメーターを指定せずに回路とターゲット QPU を Transpiler に渡します。

回路を作成して結果を確認します：

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

回路をトランスパイルして結果を確認します：

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## 利用可能なすべてのパラメーター
以下に、[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 関数で利用可能なすべてのパラメーターを示します。引数には 2 つの種類があります：コンパイルのターゲットを説明するものと、Transpiler の動作に影響を与えるものです。

`optimization_level` 以外のすべてのパラメーターはオプションです。詳細については、[Transpiler API ドキュメント](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api) を参照してください。

- `optimization_level` (int) - 回路に対して実行する最適化の度合い。範囲 (0 - 3) の整数。レベルが高いほど、より最適化された回路が生成されますが、トランスパイルに要する時間が長くなります。詳細については [Transpiler の最適化レベルの設定](set-optimization) を参照してください。

### コンパイルターゲットを説明するパラメーター：
これらの引数は、回路を実行するターゲット QPU を説明するもので、QPU のカップリングマップ（Qubit の接続性を表す）、QPU がサポートするベースゲート、ゲートのエラー率などの情報が含まれます。

これらのパラメーターの多くは、[トランスパイルでよく使用されるパラメーター](common-parameters) で詳しく説明されています。

<details>
  <summary>
    **QPU (`Backend`) パラメーター**
  </summary>

**Backend パラメーター** - `backend` を指定した場合、`target` やその他の Backend オプションを指定する必要はありません。同様に、`target` を指定した場合は、`backend` やその他の Backend オプションを指定する必要はありません。
- `backend` (Backend) - これが設定されると、Transpiler は入力回路をこのデバイス向けにコンパイルします。`coupling_map` など、この設定に影響する他のオプションが設定されている場合は、`backend` の設定を上書きします。
- `target` (Target) - Backend の Transpiler ターゲット。通常は backend 引数の一部として指定されますが、Target オブジェクトを手動で構築した場合はここで指定できます。これは `backend` のターゲットを上書きします。
- `backend_properties` (BackendProperties) - QPU から返されるプロパティで、ゲートエラー、読み出しエラー、Qubit のコヒーレンス時間などの情報を含みます。この情報を提供する QPU は `backend.properties()` を実行して確認できます。
- `timing_constraints` (Dict[str, int] | None) - 命令の時間分解能に関するハードウェアの制約（オプション）。この情報は QPU の設定から提供されます。QPU が命令の時間割り当てに制約を持たない場合、`timing_constraints` は `None` となり、調整は行われません。QPU が報告する可能性のある制約には次のものがあります：
    - `granularity`: dt 単位での最小パルスゲート分解能を表す整数値。ユーザー定義のパルスゲートは、この粒度値の倍数となる継続時間を持つ必要があります。
    - `min_length`: dt 単位での最小パルスゲート長を表す整数値。ユーザー定義のパルスゲートはこの長さより長くなければなりません。
    - `pulse_alignment`: ゲート命令の開始時刻の時間分解能を表す整数値。ゲート命令はこの値の倍数の時刻に開始する必要があります。
    - `acquire_alignment`: 測定命令の開始時刻の時間分解能を表す整数値。測定命令はこの値の倍数の時刻に開始する必要があります。
</details>

<details>
  <summary>
    **レイアウトとトポロジーのパラメーター**
  </summary>

- `basis_gates` (List[str] | None) - 展開するベースゲート名のリスト。例：['u1', 'u2', 'u3', 'cx']。`None` の場合は展開しません。
- `coupling_map` (CouplingMap | List[List[int]]) - マッピングでターゲットとする有向カップリングマップ（カスタムも可）。カップリングマップが対称の場合、両方向を指定する必要があります。以下の形式がサポートされています：
    - CouplingMap インスタンス
    - リスト - 隣接行列として指定する必要があり、各エントリが QPU でサポートされる有向 2-Qubit インタラクションをすべて指定します。例：[[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - 回路操作からパルススケジュールへのマッピング。`None` の場合、QPU の `instruction_schedule_map` が使用されます。
</details>

### Transpiler の動作に影響するパラメーター
これらのパラメーターは特定のトランスパイルステージに影響します。一部のパラメーターは複数のステージに影響する場合がありますが、わかりやすさのために 1 つのステージのみに記載しています。使用したい Qubit の `initial_layout` など引数を指定した場合、その値はそれを変更する可能性のあるすべてのパスを上書きします。つまり、手動で指定したものは Transpiler によって変更されません。特定のステージの詳細については、[Transpiler のステージ](transpiler-stages) を参照してください。

<details>
  <summary>
    **初期化ステージ**
  </summary>

- `hls_config` (HLSConfig) - `HighLevelSynthesis` 変換パスに直接渡されるオプションの設定クラス `HLSConfig`。この設定クラスを使用すると、さまざまな高レベルオブジェクトの合成アルゴリズムとそのパラメーターのリストを指定できます。
- `init_method` (str) - 初期化ステージで使用するプラグイン名。デフォルトでは外部プラグインは使用されません。インストール済みプラグインの一覧は、ステージ名引数に `init` を指定して `list_stage_plugins()` を実行することで確認できます。
- `unitary_synthesis_method` (str) - 使用するユニタリ合成メソッドの名前。デフォルトでは `default` が使用されます。インストール済みプラグインの一覧は `unitary_synthesis_plugin_names()` を実行することで確認できます。
- `unitary_synthesis_plugin_config` (dict) - ユニタリ合成プラグインに直接渡されるオプションの設定辞書。デフォルトのユニタリ合成メソッドはカスタム設定を受け付けないため、デフォルトではこの設定は効果がありません。カスタム設定の適用が必要なのは、`unitary_synthesis` 引数でユニタリ合成プラグインが指定されている場合に限ります。これは各ユニタリ合成プラグインに固有のため、このオプションの使い方についてはプラグインのドキュメントを参照してください。
</details>

<details>
  <summary>
    **レイアウトステージ**
  </summary>

- `initial_layout` (Layout | Dict | List) - 物理 Qubit 上の仮想 Qubit の初期配置。このレイアウトが `coupling_map` の制約と互換性がある場合に使用されます。Transpiler が SWAP などによって Qubit を入れ替える可能性があるため、最終的なレイアウトが同じになるとは限りません。詳細については、[初期レイアウトのセクション](common-parameters#initial-layout) を参照してください。
- `layout_method` (str) - レイアウト選択パスの名前（`default`、`dense`、`sabre`、`trivial`）。レイアウトステージで使用する外部プラグイン名を指定することもできます。インストール済みプラグインの一覧は、`stage_name` 引数に `layout` を指定して `list_stage_plugins()` を実行することで確認できます。デフォルト値は `sabre` です。
</details>

<details>
  <summary>
    **ルーティングステージ**
  </summary>

- `routing_method` (str) - ルーティングパスの名前（`basic`、`lookahead`、`default`、`sabre`、`none`）。ルーティングステージで使用する外部プラグイン名を指定することもできます。インストール済みプラグインの一覧は、`stage_name` 引数に `routing` を指定して `list_stage_plugins()` を実行することで確認できます。デフォルト値は `sabre` です。
</details>

<details>
  <summary>
    **変換ステージ**
  </summary>

- `translation_method` (str) - 変換パスの名前（`default`、`synthesis`、`translator`、`ibm_backend`、`ibm_dynamic_circuits`、`ibm_fractional`）。変換ステージで使用する外部プラグイン名を指定することもできます。インストール済みプラグインの一覧は、`stage_name` 引数に `translation` を指定して `list_stage_plugins()` を実行することで確認できます。デフォルト値は `translator` です。
</details>

<details>
  <summary>
    **最適化ステージ**
  </summary>

- `approximation_degree` (float、範囲 0-1 | None) - 回路近似に使用するヒューリスティックなダイヤル（1.0 = 近似なし、0.0 = 最大近似）。デフォルト値は 1.0 です。`None` を指定すると、報告されたエラー率に近似度が設定されます。詳細については [近似度のセクション](common-parameters#approx-degree) を参照してください。
- `optimization_method` (str) - 最適化ステージで使用するプラグイン名。デフォルトでは外部プラグインは使用されません。インストール済みプラグインの一覧は、`stage_name` 引数に `optimization` を指定して `list_stage_plugins()` を実行することで確認できます。
</details>

<details>
  <summary>
    **スケジューリングステージ**
  </summary>

- `scheduling_method` (str) - スケジューリングパスの名前。スケジューリングステージで使用する外部プラグイン名を指定することもできます。インストール済みプラグインの一覧は、`stage_name` 引数に `scheduling` を指定して `list_stage_plugins()` を実行することで確認できます。
  * 'as_soon_as_possible': Qubit リソース上でできるだけ早く命令を貪欲にスケジュールします（エイリアス：`asap`）。
  * 'as_late_as_possible': 命令を遅くスケジュールします。つまり、可能な限り Qubit を基底状態に保ちます（エイリアス：`alap`）。これがデフォルトです。
</details>

<details>
  <summary>
    **Transpiler の実行**
  </summary>

- `seed_transpiler` (int) - Transpiler の確率的部分に使用する乱数シードを設定します。
</details>

上記のパラメーターをいずれも指定しない場合、以下のデフォルト値が使用されます。詳細については、メソッドの [API リファレンスページ](../api/qiskit/transpiler_preset) を参照してください：

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>